In [0]:
import logging as l
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
l.basicConfig(level=l.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = l.getLogger(__name__)

start_time = datetime.now()
logger.info("Silver layer started")

In [0]:
try: 
    bronze_df = spark.sql("SELECT * FROM muse.bronze.job_muse_bronze_landing")

    silver_df = bronze_df.select(
        F.col("id"),
        F.col("name"),
        F.col("company.id").alias("company_id"),
        F.col("company.name").alias("company_name"),
        F.try_element_at("locations", F.lit(1))["name"].alias("location"),
        F.try_element_at("levels", F.lit(1))["name"].alias("level"),
        F.col("type"),
        F.col("model_type"),
        F.to_timestamp("publication_date").alias("publication_date"),
        F.col("refs.landing_page").alias("job_url"),
        F.col("short_name"),
        F.col("_ingestion_date")
    ).dropDuplicates(["id"])
    # display(silver_df)
    silver_df.show(5, truncate=False)
except Exception as e: 
    logger.error(f"Error: {e}")
    raise
finally:
    print("ingested into Silver layer!")
    # silver_df.printSchema()

In [0]:


silver_df.printSchema()

In [0]:
%sql
select * from muse.bronze.job_muse_bronze_landing

In [0]:
%sql 
-- create schema if not exists muse.silver 

In [0]:

%sql
-- describe catalog muse 

In [0]:
# dbutils.fs.rm("abfss://silver@musedatapipeline.dfs.core.windows.net/bronze/jobs", recurse=True)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", "abfss://silver@musedatapipeline.dfs.core.windows.net/bronze/jobs") \
    .saveAsTable("muse.silver.job_muse_silver_staging")
logger.info("ingested into Silver staging layer ")

In [0]:
%sql
-- drop table if exists muse.bronze.job_muse_silver

In [0]:
%sql

select * from muse.silver.job_muse_silver_staging;

In [0]:
%sql 
select * from muse.silver.job_muse_silver_staging

# transformations

In [0]:
%sql
select count(*) from muse.silver.job_muse_silver_staging 

In [0]:
%sql
-- create schema muse.gold

In [0]:
# %sql DESCRIBE TABLE muse.silver.job_muse_silver_main

In [0]:
# %sql 
# create table if not exists muse.silver.job_muse_silver_main
# -- using delta 
# location 'abfss://gold@musedatapipeline.dfs.core.windows.net/gold/jobs';

In [0]:
%sql 
select count(*) from muse.silver.job_muse_silver_staging

In [0]:
%sql
select count(*)  from muse.silver.job_muse_silver_main

In [0]:
# %sql
# create  table if not exists muse.silver.job_muse_silver_main
# using delta 

# location 'abfss://silver@musedatapipeline.dfs.core.windows.net/silver/main';

In [0]:
%sql 
show tables in muse.silver 

In [0]:
%sql
select count(*) from muse.silver.job_muse_silver_main

In [0]:
%sql
-- drop table if exists muse.silver.job_muse_silver_main

In [0]:
%sql
-- describe table muse.silver.job_muse_silver_main

In [0]:
%sql
-- drop table muse.silver.job_muse_silver_main


In [0]:
# %sql
# create table if not exists muse.silver.job_muse_silver_main (
# job_id BIGINT,
#   job_title STRING,
#   company_id BIGINT,
#   company_name STRING,
#   mode_of_work STRING,
#   `location-city` STRING,
#   `location-state` STRING,
#   exp_level STRING,
#   type STRING,
#   job_url STRING,
#   publication_date TIMESTAMP,
#   _ingestion_date DATE
# )
# using delta 
# location 'abfss://silver@musedatapipeline.dfs.core.windows.net/silver/main';

In [0]:
# dbutils.fs.rm("abfss://silver@musedatapipeline.dfs.core.windows.net/silver/main", recurse=True)

In [0]:
%sql
-- drop table if exists muse.silver.job_muse_silver_main

In [0]:
%sql
select * from muse.silver.job_muse_silver_main

In [0]:
%sql
-- drop view if exists muse.silver.job_muse_silver_temp

In [0]:
%sql
describe table muse.silver.job_muse_silver_main

In [0]:
%sql
create or replace table muse.silver.job_muse_silver_temp
using delta 
location 'abfss://silver@musedatapipeline.dfs.core.windows.net/silver/temp' as 
SELECT
  id as job_id,
  TRIM(name) AS job_title,
  company_id,
  TRIM(company_name) AS company_name,
  CASE 
    WHEN location LIKE '%Remote%' OR location LIKE '%Flexible%' THEN "Remote"
    ELSE "On-Site" 
  END AS mode_of_work,
  SPLIT(location, ',')[0] AS `location-city`,
  TRIM(try_element_at(SPLIT(location, ','), 2)) AS `location-state`,
  case when level like"Mid%" then '2-5 yrs' 
   when level like 'Senior%' then '5+ yrs' end as exp_level,
  type,
  job_url, 
   publication_date,
  _ingestion_date
FROM muse.silver.job_muse_silver_staging
WHERE id IS NOT NULL
AND publication_date IS NOT NULL

  

In [0]:
%sql
describe table muse.silver.job_muse_silver_temp

In [0]:
%sql
select * from muse.silver.job_muse_silver_staging

In [0]:
%sql
MERGE INTO muse.silver.job_muse_silver_main t
USING muse.silver.job_muse_silver_temp s
ON t.job_id = s.job_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
select * from muse.silver.job_muse_silver_main

In [0]:
%sql
select count(*) from muse.silver.job_muse_silver_staging
union all 
select count(*) from muse.silver.job_muse_silver_temp
union all 
select count(*) from muse.silver.job_muse_silver_main

In [0]:
%sql
SELECT id,count(*) FROM muse.silver.job_muse_silver_staging WHERE id IS NOT NULL AND publication_date IS NOT NULL
group by id having count(*)>1

In [0]:
%sql 
select * from muse.silver.job_muse_silver_main

In [0]:
%sql 
select company_name , count(*) from muse.silver.job_muse_silver_staging

group by company_name
order by count(*) desc 